# AgriWise — Complete Modeling & Proxy Pipeline V3.2

**Scope:** CALABARZON · Rice/Palay · Tomato · Red Onion · Banana

This notebook supersedes V3.1 as the recommended **single master modeling notebook**.

It is designed around the datasets that are actually available:

- PSA crop production, area planted/harvested, bearing trees and rice stocks
- PSA monthly farmgate prices
- FIES-LFS 2023 household summary and dictionary
- Monthly LFS PUF files
- optional PSA commodity-flow API

## V3.2 modeling policy

The notebook distinguishes four different things:

1. **Supply forecast** — physical production in metric tons.
2. **Farmgate-price forecast** — source price units, normally ₱/kg where applicable.
3. **Household demand proxy** — FIES expenditure-category estimation, not physical commodity demand.
4. **Quarterly demand-pressure proxy** — a derived index using FIES baseline + temporal indicators. It is an *estimated proxy*, not observed quarterly consumption.

No component is allowed to manufacture observations merely to obtain a forecast.

### FIES target mapping used

| AgriWise commodity | FIES target | Meaning in AgriWise |
|---|---|---|
| Rice | `BREAD` | Cereal/bread expenditure proxy; not rice-only |
| Tomato | `VEG` | Broad vegetable expenditure proxy |
| Red Onion | `VEG` | Broad vegetable expenditure proxy |
| Banana | `FRUIT` | Broad fruit expenditure proxy |

Tomato and Red Onion therefore share the same broad FIES target unless a more granular consumption source becomes available.

### Important V3.2 fixes

- `0` farmgate prices are treated as **missing observations**, not real ₱0 prices.
- duplicate PSA wide rows are consolidated by geolocation/commodity/period.
- `SEQ_NO` and other identifiers are excluded from demand-model predictors.
- target-derived expenditure totals such as `FOOD_HOME`, `FOOD`, and `TOTEX` are excluded from demand-model predictors to reduce leakage.
- survey-weighted FIES province aggregation uses `RFACT`.
- supply models may use lagged area/yield/tree/stock/price information where available.
- price models may use lagged supply/area/yield information.
- demand models are evaluated as **cross-sectional proxy estimators**, not falsely as quarterly observed-demand forecasters.


In [1]:

# Optional installation if your environment is missing packages:
# %pip install -q pandas numpy scikit-learn xgboost joblib openpyxl requests matplotlib

from pathlib import Path
from datetime import datetime
import json, re, zipfile, shutil, warnings, math

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

COMMODITIES = ["rice", "tomato", "red_onion", "banana"]
CALABARZON = ["Batangas", "Cavite", "Laguna", "Quezon", "Rizal"]

MONTHS = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12,
}

# FIES RPROV codes from the supplied 2023 valueset.
FIES_PROVINCE_MAP = {
    1000: "Batangas",
    2100: "Cavite",
    3400: "Laguna",
    5600: "Quezon",
    5624: "Quezon",   # Lucena City is grouped with Quezon for AgriWise province-level mapping.
    5800: "Rizal",
}

DEMAND_TARGETS = {
    "rice": "BREAD",
    "tomato": "VEG",
    "red_onion": "VEG",
    "banana": "FRUIT",
}

# Keep these explicit. They are economic/demographic predictors, not target-derived food totals.
DEMAND_NUMERIC_FEATURES = [
    "FSIZE", "URB", "TOINC", "PERCAPITA", "EAINC", "WAGES",
    "REG_SAL", "SEASON_SAL", "NET_CFG", "NETSHARE",
    "CASH_ABROAD", "CASH_DOMESTIC", "PENSION", "RENTALS_REC",
    "INTEREST", "NET_LPR", "NET_FISH", "NET_FOR", "NET_RET",
    "NET_MFG", "NET_TRANS",
]
DEMAND_CATEGORICAL_FEATURES = ["RPROV"]

# --- V3.3 additions: weighted demand training, an XGBoost benchmark race, and
# EF_agriwise forecast-service artifact-contract compatibility. ---

# Matches EF_agriwise/ml/demand/pipeline/train_quarterly_101.py exactly, so the
# GradientBoosting-vs-XGBoost demand race is an apples-to-apples comparison
# against the team's established FIES/XGBoost reference pattern (ross'_work).
DEMAND_XGB_PARAMETERS = {
    "n_estimators": 900,
    "learning_rate": 0.01,
    "max_depth": 2,
    "min_child_weight": 12,
    "subsample": 0.65,
    "colsample_bytree": 0.50,
    "gamma": 0.05,
    "reg_alpha": 0.50,
    "reg_lambda": 5.0,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Must match EF_agriwise/ml/forecasting/artifact_registry.py::slugify_commodity
# exactly -- this is the filename convention the forecast-service registry expects.
def slugify_commodity(commodity):
    return commodity.strip().lower().replace(" ", "_")

# Must match EF_agriwise/ml/forecasting/domain.py::COMMODITIES exactly (Title Case).
COMMODITY_TITLE_CASE = {
    "rice": "Rice",
    "tomato": "Tomato",
    "red_onion": "Red Onion",
    "banana": "Banana",
}

# The FIES VEG demand model is shared across Tomato and Red Onion, so the one
# trained artifact is promoted into two separate commodity-keyed bundles.
DEMAND_ARTIFACT_COMMODITIES = {
    "rice": ["Rice"],
    "vegetable_shared": ["Tomato", "Red Onion"],
    "banana": ["Banana"],
}

def find_ef_agriwise_root():
    here = Path.cwd()
    for candidate in [here / "EF_agriwise", here.parent / "EF_agriwise", here.parent.parent / "EF_agriwise"]:
        if (candidate / "ml" / "forecasting" / "artifact_registry.py").exists():
            return candidate.resolve()
    return None

# Where trained artifacts are promoted so EF_agriwise/ml/forecasting/artifact_registry.py
# can load them directly. Resolved relative to this notebook's working directory;
# promotion is skipped gracefully (see the final cell) if EF_agriwise isn't found.
EF_AGRIWISE_ROOT = find_ef_agriwise_root()
EF_ARTIFACTS_ROOT = (EF_AGRIWISE_ROOT / "ml" / "artifacts") if EF_AGRIWISE_ROOT else None
if EF_ARTIFACTS_ROOT is None:
    print("EF_agriwise not found relative to this notebook; artifact promotion will be skipped at the end of the run.")
else:
    print("EF_agriwise artifacts will be promoted to:", EF_ARTIFACTS_ROOT)

ARTIFACT_ROOT = Path("artifacts_v3_2")
for d in [
    ARTIFACT_ROOT / "supply",
    ARTIFACT_ROOT / "price",
    ARTIFACT_ROOT / "demand",
    ARTIFACT_ROOT / "prepared",
    ARTIFACT_ROOT / "reports",
    ARTIFACT_ROOT / "config",
]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifact root:", ARTIFACT_ROOT.resolve())


EF_agriwise artifacts will be promoted to: C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts
Artifact root: C:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\artifacts_v3_2


## 1. Locate/extract the supplied datasets

In [2]:

# The notebook supports either extracted folders or ZIP files.
# Put it inside your AgriDataSet folder, repository root, or point BASE_DIR manually.

BASE_DIR = Path.cwd()

def find_one(patterns, roots=None):
    roots = roots or [BASE_DIR, BASE_DIR / "AgriDataSet", BASE_DIR / "data", BASE_DIR / "ml"]
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            hits = list(root.rglob(pat))
            if hits:
                return hits[0]
    return None

def extract_zip_if_needed(zip_path, label):
    if zip_path is None:
        return None
    dest = BASE_DIR / ".agriwise_cache" / label
    marker = dest / ".complete"
    if not marker.exists():
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
        marker.write_text("ok", encoding="utf-8")
    return dest

crop_dir = find_one(["psa_openstat_crops"])
price_dir = find_one(["psa_price_downloads"])
lfs_root = find_one(["lfsFiles"])

if crop_dir is None or not crop_dir.is_dir():
    crop_zip = find_one(["psa_openstat_crops*.zip"])
    cache = extract_zip_if_needed(crop_zip, "crops")
    crop_dir = find_one(["psa_openstat_crops"], [cache]) if cache else None

if price_dir is None or not price_dir.is_dir():
    price_zip = find_one(["psa_price_downloads*.zip"])
    cache = extract_zip_if_needed(price_zip, "prices")
    price_dir = find_one(["psa_price_downloads"], [cache]) if cache else None

if lfs_root is None or not lfs_root.is_dir():
    lfs_zip = find_one(["lfsFiles*.zip"])
    cache = extract_zip_if_needed(lfs_zip, "lfs")
    lfs_root = find_one(["lfsFiles"], [cache]) if cache else None

print("crop_dir :", crop_dir)
print("price_dir:", price_dir)
print("lfs_root :", lfs_root)

assert crop_dir is not None, "PSA crop dataset folder/ZIP not found."
assert price_dir is not None, "PSA price dataset folder/ZIP not found."
assert lfs_root is not None, "FIES/LFS dataset folder/ZIP not found."


crop_dir : C:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\psa_openstat_crops
price_dir: C:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\psa_price_downloads
lfs_root : C:\Users\jramo\Downloads\DOST_Agriwise\trainings_file\lfsFiles


## 2. Dataset inventory and structural audit

In [3]:

def csv_shape(path):
    try:
        sample = pd.read_csv(path, nrows=5, low_memory=False)
        # Count rows cheaply only when needed later; inventory uses columns + file size.
        return len(sample.columns)
    except Exception:
        return np.nan

inventory = []
for family, root in [("crops", crop_dir), ("prices", price_dir), ("fies_lfs", lfs_root)]:
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".csv", ".xlsx"}:
            inventory.append({
                "family": family,
                "file": p.name,
                "size_mb": round(p.stat().st_size / (1024**2), 2),
                "columns_sampled": csv_shape(p) if p.suffix.lower() == ".csv" else np.nan,
            })

inventory_df = pd.DataFrame(inventory).sort_values(["family", "file"])
display(inventory_df)
inventory_df.to_csv(ARTIFACT_ROOT / "reports" / "dataset_inventory.csv", index=False)


,family,file,size_mb,columns_sampled
0,crops,"Fruit Crops- Area Planted-Harvested, by Region...",24.88,121.0
1,crops,Fruit Crops- Number of Bearing Trees-Hills-Vin...,22.56,121.0
2,crops,"Fruit Crops- Volume of Production, by Region, ...",25.53,121.0
3,crops,Major Crops- Number of Bearing Trees-Hills-Vin...,0.22,121.0
4,crops,Palay and Corn- Area Harvested in Hectares by ...,1.70,282.0
...,...,...,...,...
12,prices,"Cereals- Farmgate Prices by Geolocation, Commo...",3.44,223.0
13,prices,"Condiments- Farmgate Prices by Geolocation, Co...",6.60,223.0
14,prices,Fruit Vegetables- Farmgate Prices by Geolocati...,8.20,223.0
15,prices,"Fruits- Farmgate Prices by Commodity, by Regio...",30.71,223.0


## 3. Shared parsing and validation helpers

In [4]:

def clean_geo(v):
    return re.sub(r"^\.+", "", str(v)).strip()

def numeric_clean(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .replace({"..": np.nan, "...": np.nan, "nan": np.nan, "": np.nan}),
        errors="coerce",
    )

def parse_quarter_column(col):
    m = re.search(r"(\d{4}).*Quarter\s*([1-4])", str(col), re.I)
    if not m:
        return None
    return pd.Timestamp(int(m.group(1)), (int(m.group(2))-1)*3 + 1, 1)

def parse_month_column(col):
    m = re.match(r"^(\d{4})\s+(" + "|".join(MONTHS) + r")$", str(col))
    if not m:
        return None
    return pd.Timestamp(int(m.group(1)), MONTHS[m.group(2)], 1)

def regression_metrics(y, pred, seasonal_naive=None):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = np.isfinite(y) & np.isfinite(pred)
    y, pred = y[mask], pred[mask]
    if len(y) == 0:
        return {}
    err = pred - y
    mae = mean_absolute_error(y, pred)
    mse = mean_squared_error(y, pred)
    rmse = math.sqrt(mse)
    r2 = r2_score(y, pred) if len(y) > 1 and np.nanstd(y) > 1e-12 else np.nan
    nz = np.abs(y) > 1e-12
    mape = np.mean(np.abs(err[nz] / y[nz])) * 100 if nz.any() else np.nan
    denom = np.abs(y) + np.abs(pred)
    sm = denom > 1e-12
    smape = np.mean(2 * np.abs(err[sm]) / denom[sm]) * 100 if sm.any() else np.nan
    wape = np.sum(np.abs(err)) / np.sum(np.abs(y)) * 100 if np.sum(np.abs(y)) > 1e-12 else np.nan
    bias = np.mean(err)
    mase = np.nan
    if seasonal_naive is not None:
        naive = np.asarray(seasonal_naive, dtype=float)[mask]
        naive_mae = np.nanmean(np.abs(y - naive))
        if np.isfinite(naive_mae) and naive_mae > 1e-12:
            mase = mae / naive_mae
    return {
        "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2,
        "MAPE": mape, "sMAPE": smape, "WAPE": wape,
        "MASE": mase, "Bias": bias, "N": len(y),
        "ZeroRatePct": float(np.mean(np.isclose(y, 0))*100),
        "NonZeroN": int(np.sum(~np.isclose(y, 0))),
    }

def weighted_regression_metrics(y, pred, weights):
    """Survey-weighted metrics, matching EF_agriwise/ml/demand/pipeline's
    metrics() function (sample_weight=RFACT) so demand-proxy numbers are
    directly comparable to the team's FIES/XGBoost reference models."""
    y_raw = np.asarray(y, dtype=float)
    pred_raw = np.asarray(pred, dtype=float)
    w_raw = np.asarray(weights, dtype=float)
    nonfinite_predictions = int((~np.isfinite(pred_raw)).sum())
    negative_predictions = int((pred_raw < 0).sum())
    mask = np.isfinite(y_raw) & np.isfinite(pred_raw) & np.isfinite(w_raw) & (w_raw > 0)
    y, pred, w = y_raw[mask], pred_raw[mask], w_raw[mask]
    if len(y) == 0:
        return {}
    actual_mean = float(np.average(y, weights=w))
    mae = float(mean_absolute_error(y, pred, sample_weight=w))
    rmse = float(math.sqrt(mean_squared_error(y, pred, sample_weight=w)))
    r2 = float(r2_score(y, pred, sample_weight=w)) if len(y) > 1 else np.nan
    actual_total = float(np.sum(y * w))
    predicted_total = float(np.sum(pred * w))
    return {
        "r2": r2, "mae": mae, "rmse": rmse,
        "normalized_mae_percent": (mae / actual_mean * 100) if actual_mean else np.nan,
        "aggregate_bias_percent": ((predicted_total - actual_total) / actual_total * 100) if actual_total else np.nan,
        "target_mean": actual_mean, "prediction_mean": float(np.average(pred, weights=w)),
        "n": int(len(y)), "negative_predictions": negative_predictions,
        "nonfinite_predictions": nonfinite_predictions,
    }

def demand_verdict(weighted_r2, wape):
    """Realistic, documented tiers for cross-sectional FIES demand-proxy models --
    replaces the previous unconditional 'USABLE_PROXY' label. Bands are deliberately
    conservative: FIES category proxies are noisy household survey data, not an
    observed commodity time series."""
    if not np.isfinite(weighted_r2) and not np.isfinite(wape):
        return "INSUFFICIENT_DATA", "no valid holdout error"
    if (np.isfinite(weighted_r2) and weighted_r2 >= 0.45) or (np.isfinite(wape) and wape <= 30):
        return "USABLE_PROXY", "weighted R2 and/or WAPE meet the usable-proxy bar"
    if (np.isfinite(weighted_r2) and weighted_r2 >= 0.20) or (np.isfinite(wape) and wape <= 55):
        return "INDICATIVE_PROXY", "weak explanatory power; use for relative/directional signal, not precise point forecasts"
    return "INSUFFICIENT_DATA", "forecast errors too high to support even an indicative proxy"

def add_panel_lags(df, seasonal_period, lags):
    x = df.sort_values(["geolocation", "date"]).copy()
    g = x.groupby("geolocation", group_keys=False)["target"]
    for lag in lags:
        x[f"lag_{lag}"] = g.shift(lag)
    x["roll_mean_3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    x["roll_std_3"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
    x["season_sin"] = np.sin(2*np.pi*x["season"] / seasonal_period)
    x["season_cos"] = np.cos(2*np.pi*x["season"] / seasonal_period)
    return x


## 4. Build supply and agronomic auxiliary panels

In [5]:

CROP_FILES = {p.name: p for p in crop_dir.rglob("*.csv")}

def wide_quarter_panel(path, commodity_col, commodity_pattern, value_name):
    df = pd.read_csv(path, low_memory=False)
    df["_geo"] = df["Geolocation"].map(clean_geo)
    mask = df[commodity_col].astype(str).str.contains(commodity_pattern, case=False, regex=True, na=False)
    df = df[mask & df["_geo"].isin(CALABARZON)].copy()

    rows = []
    for c in df.columns:
        dt = parse_quarter_column(c)
        if dt is None:
            continue
        vals = numeric_clean(df[c])
        tmp = pd.DataFrame({
            "geolocation": df["_geo"],
            "date": dt,
            value_name: vals,
        })
        rows.append(tmp)

    if not rows:
        return pd.DataFrame()

    long = pd.concat(rows, ignore_index=True)
    # Duplicated wide PSA blocks are consolidated. Missing is not converted to zero.
    long = (
        long.groupby(["geolocation", "date"], as_index=False)[value_name]
            .mean()
            .sort_values(["geolocation", "date"])
    )
    return long

palay_prod = wide_quarter_panel(
    CROP_FILES[[n for n in CROP_FILES if n.startswith("Palay and Corn- Volume")][0]],
    "Ecosystem/Croptype", r"Palay", "production"
)
palay_area = wide_quarter_panel(
    CROP_FILES[[n for n in CROP_FILES if n.startswith("Palay and Corn- Area")][0]],
    "Ecosystem/Croptype", r"Palay", "area"
)
veg_prod_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Vegetables and Root Crops- Volume")][0]]
veg_area_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Vegetables and Root Crops- Area")][0]]
fruit_prod_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Fruit Crops- Volume")][0]]
fruit_area_file = CROP_FILES[[n for n in CROP_FILES if n.startswith("Fruit Crops- Area")][0]]

tomato_prod = wide_quarter_panel(veg_prod_file, "Crop", r"^Tomato$", "production")
tomato_area = wide_quarter_panel(veg_area_file, "Crop", r"^Tomato$", "area")
onion_prod = wide_quarter_panel(veg_prod_file, "Crop", r"Onion.*red|Red.*onion|Onion", "production")
onion_area = wide_quarter_panel(veg_area_file, "Crop", r"Onion.*red|Red.*onion|Onion", "area")
banana_prod = wide_quarter_panel(fruit_prod_file, "Crop", r"^Banana$", "production")
banana_area = wide_quarter_panel(fruit_area_file, "Crop", r"^Banana$", "area")

supply_raw = {
    "rice": palay_prod,
    "tomato": tomato_prod,
    "red_onion": onion_prod,
    "banana": banana_prod,
}
area_raw = {
    "rice": palay_area,
    "tomato": tomato_area,
    "red_onion": onion_area,
    "banana": banana_area,
}

supply_panels = {}
for commodity in COMMODITIES:
    p = supply_raw[commodity].rename(columns={"production":"target"})
    a = area_raw[commodity]
    x = p.merge(a, on=["geolocation","date"], how="left")
    x["yield_proxy"] = np.where(x["area"] > 0, x["target"] / x["area"], np.nan)
    x["year"] = x["date"].dt.year
    x["season"] = x["date"].dt.quarter
    supply_panels[commodity] = x

supply_audit = pd.concat([
    pd.DataFrame({
        "commodity":[c],
        "rows":[len(x)],
        "date_min":[x["date"].min()],
        "date_max":[x["date"].max()],
        "missing_target_pct":[x["target"].isna().mean()*100],
        "zero_target_pct":[np.isclose(x["target"].fillna(np.nan),0).mean()*100],
        "locations":[x["geolocation"].nunique()],
    })
    for c,x in supply_panels.items()
], ignore_index=True)
display(supply_audit)
supply_audit.to_csv(ARTIFACT_ROOT / "reports" / "supply_data_audit.csv", index=False)


,commodity,rows,date_min,date_max,missing_target_pct,zero_target_pct,locations
0,rice,800,1987-01-01,2026-10-01,1.250000,0.000000,5
1,tomato,340,2010-01-01,2026-10-01,2.941176,10.588235,5
2,red_onion,340,2010-01-01,2026-10-01,2.941176,45.000000,5
3,banana,340,2010-01-01,2026-10-01,2.941176,0.000000,5


## 5. Build monthly farmgate-price panels — **zero-price fix included**

In [6]:

PRICE_FILES = {p.name: p for p in price_dir.rglob("*.csv")}

PRICE_SPECS = {
    "rice": (
        [n for n in PRICE_FILES if n.startswith("Cereals- Farmgate")][0],
        r"Palay \[Paddy\]",
    ),
    "tomato": (
        [n for n in PRICE_FILES if n.startswith("Fruit Vegetables- Farmgate")][0],
        r"^Tomato$",
    ),
    "red_onion": (
        [n for n in PRICE_FILES if n.startswith("Condiments- Farmgate")][0],
        r"Onion, red",
    ),
    "banana": (
        [n for n in PRICE_FILES if n.startswith("Fruits- Farmgate")][0],
        r"Banana",
    ),
}

def build_price_panel(path, pattern):
    df = pd.read_csv(path, low_memory=False)
    df["_geo"] = df["Geolocation"].map(clean_geo)
    df = df[
        df["Commodity"].astype(str).str.contains(pattern, case=False, regex=True, na=False)
        & df["_geo"].isin(CALABARZON)
    ].copy()

    rows = []
    for c in df.columns:
        dt = parse_month_column(c)
        if dt is None:
            continue
        val = numeric_clean(df[c])
        # Critical fix: a real farmgate price cannot be represented as zero here.
        # PSA tables use explicit zeros in many sparse rows; treat <=0 as unavailable.
        val = val.mask(val <= 0)
        rows.append(pd.DataFrame({
            "geolocation": df["_geo"],
            "date": dt,
            "price": val,
            "variant": df["Commodity"].astype(str),
        }))

    long = pd.concat(rows, ignore_index=True)

    # Multiple varieties can exist. First collapse duplicates within variety/date,
    # then take median across available varieties for a robust commodity-level price.
    long = (
        long.groupby(["geolocation","date","variant"], as_index=False)["price"].mean()
    )
    panel = (
        long.groupby(["geolocation","date"], as_index=False)["price"].median()
            .rename(columns={"price":"target"})
            .sort_values(["geolocation","date"])
    )
    panel["year"] = panel["date"].dt.year
    panel["season"] = panel["date"].dt.month
    return panel

price_panels = {
    c: build_price_panel(PRICE_FILES[fname], patt)
    for c,(fname,patt) in PRICE_SPECS.items()
}

price_audit_rows = []
for c,x in price_panels.items():
    expected = x["geolocation"].nunique() * x["date"].nunique()
    price_audit_rows.append({
        "commodity": c,
        "rows_with_grid_keys": len(x),
        "nonmissing_price_n": int(x["target"].notna().sum()),
        "missing_price_pct": float(x["target"].isna().mean()*100),
        "zero_price_after_cleaning_n": int(np.isclose(x["target"].fillna(np.nan),0).sum()),
        "date_min": x["date"].min(),
        "date_max": x["date"].max(),
        "locations": x["geolocation"].nunique(),
    })

price_audit = pd.DataFrame(price_audit_rows)
display(price_audit)
price_audit.to_csv(ARTIFACT_ROOT / "reports" / "price_data_audit.csv", index=False)

for c,x in price_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_price_panel.csv", index=False)


,commodity,rows_with_grid_keys,nonmissing_price_n,missing_price_pct,zero_price_after_cleaning_n,date_min,date_max,locations
0,rice,1020,988,3.137255,0,2010-01-01,2026-12-01,5
1,tomato,1020,798,21.764706,0,2010-01-01,2026-12-01,5
2,red_onion,1020,48,95.294118,0,2010-01-01,2026-12-01,5
3,banana,1020,990,2.941176,0,2010-01-01,2026-12-01,5


### Why Red Onion price previously failed

The supplied raw condiments table contains many explicit `0.00` monthly values for red-onion rows.  
Those must **not** be trained as actual farmgate prices. V3.2 converts non-positive prices to missing before modeling.

This will usually improve Red Onion diagnostics substantially, although the series may still be too sparse for province-level forecasting.


## 6. Join supply ↔ price and create leakage-safe lag features

In [7]:

def quarterly_to_monthly_lagged(qdf, value_col, out_col):
    q = qdf[["geolocation","date",value_col]].copy()
    q["quarter_end"] = q["date"] + pd.offsets.QuarterEnd(0)
    q = q.rename(columns={value_col:out_col})
    return q[["geolocation","quarter_end",out_col]]

# Supply features: area/yield plus lagged monthly price aggregated to quarter.
for c in COMMODITIES:
    s = supply_panels[c].copy()
    p = price_panels[c].copy()
    p["qdate"] = p["date"].dt.to_period("Q").dt.start_time
    pq = p.groupby(["geolocation","qdate"],as_index=False)["target"].mean().rename(
        columns={"qdate":"date","target":"quarter_price"}
    )
    s = s.merge(pq, on=["geolocation","date"], how="left")
    s["quarter_price_lag1"] = s.groupby("geolocation")["quarter_price"].shift(1)
    supply_panels[c] = add_panel_lags(s, 4, [1,2,4,8])

# Price features: map the most recently completed/current-quarter supply context.
for c in COMMODITIES:
    p = price_panels[c].copy()
    s = supply_panels[c][["geolocation","date","target","area","yield_proxy"]].copy()
    s = s.rename(columns={"target":"quarter_supply","date":"qdate"})
    p["qdate"] = p["date"].dt.to_period("Q").dt.start_time
    p = p.merge(s, on=["geolocation","qdate"], how="left")
    # Lag exogenous agricultural values by one quarter-equivalent merge context
    p["quarter_supply_lag"] = p.groupby("geolocation")["quarter_supply"].shift(3)
    p["area_lag"] = p.groupby("geolocation")["area"].shift(3)
    p["yield_proxy_lag"] = p.groupby("geolocation")["yield_proxy"].shift(3)
    price_panels[c] = add_panel_lags(p, 12, [1,2,3,6,12])

for c,x in supply_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_supply_features.csv", index=False)
for c,x in price_panels.items():
    x.to_csv(ARTIFACT_ROOT / "prepared" / f"{c}_price_features.csv", index=False)


## 7. Time-series model selection and rolling-origin validation

In [8]:

def feature_columns(kind, df):
    base = ["season_sin","season_cos","roll_mean_3","roll_std_3"]
    lags = [c for c in df.columns if c.startswith("lag_")]
    if kind == "supply":
        extra = [c for c in ["area","yield_proxy","quarter_price_lag1"] if c in df]
    else:
        extra = [c for c in ["quarter_supply_lag","area_lag","yield_proxy_lag"] if c in df]
    return lags + base + extra

def make_candidate(name):
    if name == "hist_gradient_boosting":
        return HistGradientBoostingRegressor(
            max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
            l2_regularization=1.0, random_state=RANDOM_STATE
        )
    if name == "random_forest":
        return RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_STATE
        )
    raise ValueError(name)

def chronological_split(df, holdout_periods):
    dates = np.array(sorted(df["date"].dropna().unique()))
    if len(dates) <= holdout_periods + 4:
        return None, None
    cutoff = dates[-holdout_periods]
    return df[df["date"] < cutoff].copy(), df[df["date"] >= cutoff].copy()

def seasonal_naive_predict(train, test, seasonal_period):
    hist = pd.concat([
        train[["geolocation","date","target"]],
        test[["geolocation","date","target"]].assign(target=np.nan)
    ]).sort_values(["geolocation","date"])
    hist["naive"] = hist.groupby("geolocation")["target"].shift(seasonal_period)
    return hist.loc[hist["date"].isin(test["date"]), ["geolocation","date","naive"]]

def evaluate_candidate(train, test, kind, model_name, seasonal_period):
    feats = feature_columns(kind, train)
    required = f"lag_{seasonal_period}"
    tr = train.dropna(subset=["target", required]).copy()
    te = test.dropna(subset=["target", required]).copy()
    if len(tr) < 40 or len(te) == 0:
        return None

    naive = te[required].to_numpy()

    if model_name == "seasonal_naive":
        pred = naive.copy()
        model = None
    else:
        use = [c for c in feats if c in tr.columns]
        pre = Pipeline([("imp",SimpleImputer(strategy="median"))])
        model = Pipeline([
            ("prep", pre),
            ("model", make_candidate(model_name))
        ])
        model.fit(tr[use], tr["target"])
        pred = model.predict(te[use])

    metrics = regression_metrics(te["target"], pred, seasonal_naive=naive)
    return {"model":model, "metrics":metrics, "pred":pred, "test":te, "features":feats}

def rolling_validate(df, kind, model_name, seasonal_period, n_splits=6):
    dates = np.array(sorted(df["date"].dropna().unique()))
    min_train = max(seasonal_period*3, 12)
    if len(dates) < min_train + n_splits:
        return pd.DataFrame()

    split_points = np.linspace(min_train, len(dates)-1, n_splits, dtype=int)
    rows = []
    for i,sp in enumerate(split_points,1):
        train_dates = dates[:sp]
        test_dates = dates[sp:sp+1]
        tr = df[df["date"].isin(train_dates)]
        te = df[df["date"].isin(test_dates)]
        result = evaluate_candidate(tr, te, kind, model_name, seasonal_period)
        if result is None:
            continue
        rows.append({"split":i,"test_date":test_dates[0],**result["metrics"]})
    return pd.DataFrame(rows)

def deployment_verdict(metrics, rolling, improvement):
    sm = metrics.get("sMAPE", np.nan)
    mase = metrics.get("MASE", np.nan)
    roll_med = rolling["sMAPE"].median() if not rolling.empty else np.nan
    roll_worst = rolling["sMAPE"].max() if not rolling.empty else np.nan

    if not np.isfinite(sm):
        return "INSUFFICIENT_DATA", "no valid holdout error"

    # A model that is not even better than the seasonal-naive baseline (MASE >= 1)
    # should never be reported as an unqualified PASS, however low its sMAPE looks.
    mase_ok = not np.isfinite(mase) or mase < 1.0

    if sm <= 30 and (not np.isfinite(roll_med) or roll_med <= 35):
        if np.isfinite(roll_worst) and roll_worst > 55:
            return "CAUTION", "rolling variability is elevated"
        if not mase_ok:
            return "CAUTION", f"MASE={mase:.2f} indicates no improvement over the seasonal-naive baseline"
        return "PASS", ""
    if sm <= 50 and (not np.isfinite(roll_med) or roll_med <= 50):
        return "CAUTION", "moderate forecast error"
    return "INSUFFICIENT_DATA", "forecast errors remain too high"

def train_time_series_family(panels, kind):
    holdout_periods = 4 if kind == "supply" else 12
    seasonal_period = 4 if kind == "supply" else 12
    all_metrics, all_roll, verdicts = [], [], []

    for commodity,df in panels.items():
        df = df.copy().dropna(subset=["target"])
        train,test = chronological_split(df, holdout_periods)
        if train is None:
            verdicts.append({"commodity":commodity,"kind":kind,"verdict":"INSUFFICIENT_DATA","reason":"too few periods"})
            continue

        candidates = {}
        for name in ["seasonal_naive","hist_gradient_boosting","random_forest"]:
            res = evaluate_candidate(train,test,kind,name,seasonal_period)
            if res:
                candidates[name] = res

        if not candidates:
            verdicts.append({"commodity":commodity,"kind":kind,"verdict":"INSUFFICIENT_DATA","reason":"no candidate could be trained"})
            continue

        # Primary selector: sMAPE. MASE/WAPE are retained as secondary evidence.
        selected_name = min(
            candidates,
            key=lambda n: candidates[n]["metrics"].get("sMAPE", np.inf)
        )
        selected = candidates[selected_name]
        baseline = candidates.get("seasonal_naive")
        base_sm = baseline["metrics"].get("sMAPE",np.nan) if baseline else np.nan
        sel_sm = selected["metrics"].get("sMAPE",np.nan)
        improvement = ((base_sm-sel_sm)/base_sm*100) if np.isfinite(base_sm) and base_sm > 0 else 0.0

        rolling = rolling_validate(df,kind,selected_name,seasonal_period)
        verdict,reason = deployment_verdict(selected["metrics"], rolling, improvement)

        # Refit selected ML model on all valid rows for artifact export.
        feats = selected["features"]
        valid = df.dropna(subset=["target",f"lag_{seasonal_period}"]).copy()
        final_model = None
        if selected_name != "seasonal_naive":
            final_model = Pipeline([
                ("prep",Pipeline([("imp",SimpleImputer(strategy="median"))])),
                ("model",make_candidate(selected_name)),
            ])
            final_model.fit(valid[feats],valid["target"])

        artifact = {
            "schema_version":"3.2",
            "created_utc":datetime.utcnow().isoformat()+"Z",
            "commodity":COMMODITY_TITLE_CASE.get(commodity, commodity),
            "model_id":f"{commodity}-{kind}-v1",
            "kind":kind,
            "strategy":selected_name,
            "model":final_model,
            "features":feats,
            "seasonal_period":seasonal_period,
            "metrics":selected["metrics"],
            "baseline_metrics":baseline["metrics"] if baseline else {},
            "improvement_vs_naive_pct":improvement,
            "verdict":verdict,
            "reason":reason,
            "limitations":[
                f"{kind.capitalize()} time-series estimate selected by lowest sMAPE among seasonal-naive/HistGB/RandomForest candidates.",
            ],
            "history_tail":valid.groupby("geolocation").tail(seasonal_period*2)[
                ["geolocation","date","target"]
            ].to_dict("records"),
        }
        joblib.dump(artifact, ARTIFACT_ROOT / kind / f"{commodity}.joblib")

        for name,res in candidates.items():
            all_metrics.append({
                "commodity":commodity,"kind":kind,"model":name,
                **res["metrics"],
            })
        if not rolling.empty:
            rolling["commodity"]=commodity
            rolling["kind"]=kind
            rolling["model"]=selected_name
            all_roll.append(rolling)

        verdicts.append({
            "commodity":commodity,"kind":kind,"selected_model":selected_name,
            **selected["metrics"],
            "baseline_sMAPE":base_sm,
            "improvement_vs_naive_pct":improvement,
            "rolling_median_sMAPE":rolling["sMAPE"].median() if not rolling.empty else np.nan,
            "rolling_worst_sMAPE":rolling["sMAPE"].max() if not rolling.empty else np.nan,
            "verdict":verdict,"reason":reason,
        })
        print(kind, commodity, "=>", selected_name, verdict, reason)

    return (
        pd.DataFrame(all_metrics),
        pd.concat(all_roll,ignore_index=True) if all_roll else pd.DataFrame(),
        pd.DataFrame(verdicts),
    )


In [9]:

supply_metrics, supply_rolling, supply_verdicts = train_time_series_family(supply_panels, "supply")
price_metrics, price_rolling, price_verdicts = train_time_series_family(price_panels, "price")

supply_metrics.to_csv(ARTIFACT_ROOT / "reports" / "supply_model_metrics.csv", index=False)
supply_rolling.to_csv(ARTIFACT_ROOT / "reports" / "supply_rolling_validation.csv", index=False)
supply_verdicts.to_csv(ARTIFACT_ROOT / "reports" / "supply_deployment_verdicts.csv", index=False)

price_metrics.to_csv(ARTIFACT_ROOT / "reports" / "price_model_metrics.csv", index=False)
price_rolling.to_csv(ARTIFACT_ROOT / "reports" / "price_rolling_validation.csv", index=False)
price_verdicts.to_csv(ARTIFACT_ROOT / "reports" / "price_deployment_verdicts.csv", index=False)

display(supply_verdicts)
display(price_verdicts)


supply rice => hist_gradient_boosting CAUTION rolling variability is elevated


supply tomato => random_forest CAUTION rolling variability is elevated


supply red_onion => seasonal_naive INSUFFICIENT_DATA forecast errors remain too high


supply banana => seasonal_naive CAUTION MASE=1.00 indicates no improvement over the seasonal-naive baseline


price rice => hist_gradient_boosting PASS 


price tomato => random_forest CAUTION rolling variability is elevated


price banana => random_forest PASS 


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,Bias,N,ZeroRatePct,NonZeroN,baseline_sMAPE,improvement_vs_naive_pct,rolling_median_sMAPE,rolling_worst_sMAPE,verdict,reason
0,rice,supply,hist_gradient_boosting,739.047069,3.680103e+06,1918.359430,0.983102,5.416628,5.426247,6.170531,...,-455.211612,20,0.0,20,22.626605,76.018288,7.525730,58.359334,CAUTION,rolling variability is elevated
1,tomato,supply,random_forest,106.923047,7.352388e+04,271.152865,0.969183,27.248317,21.473553,14.063160,...,86.180082,20,0.0,20,28.999335,25.951568,26.481272,91.425107,CAUTION,rolling variability is elevated
2,red_onion,supply,seasonal_naive,1.261071,8.569416e+00,2.927357,0.515448,1244.634380,64.745796,50.477470,...,-0.190214,20,25.0,15,64.745796,0.000000,33.139871,88.714852,INSUFFICIENT_DATA,forecast errors remain too high
3,banana,supply,seasonal_naive,572.249000,1.307150e+06,1143.306509,0.934330,14.704234,18.967574,12.870320,...,-274.278000,20,0.0,20,18.967574,0.000000,7.071017,24.433028,CAUTION,MASE=1.00 indicates no improvement over the se...


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,Bias,N,ZeroRatePct,NonZeroN,baseline_sMAPE,improvement_vs_naive_pct,rolling_median_sMAPE,rolling_worst_sMAPE,verdict,reason
0,rice,price,hist_gradient_boosting,1.833069,6.606344,2.570281,0.676917,10.084788,10.335733,10.400240,...,-0.574450,59.0,0.0,59.0,35.286311,70.708945,5.738461,7.285104,PASS,
1,tomato,price,random_forest,14.330404,510.184494,22.587264,0.060465,38.352551,29.455760,32.377939,...,2.365037,45.0,0.0,45.0,47.790878,38.365309,32.184053,57.352353,CAUTION,rolling variability is elevated
2,red_onion,price,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_DATA,no candidate could be trained
3,banana,price,random_forest,3.821376,28.317304,5.321401,-0.096755,15.204443,14.447917,14.905999,...,0.048369,60.0,0.0,60.0,20.874724,30.787504,5.777772,17.560057,PASS,


## 8. Load FIES 2023 household-summary data and valueset

In [10]:

fies_summary_path = next(iter(lfs_root.rglob("FIES-LFS-houseHoldSummary.CSV")), None)
fies_xlsx_path = next(iter(lfs_root.rglob("fies_lfs.xlsx")), None)

assert fies_summary_path is not None, "FIES household summary not found."

fies = pd.read_csv(fies_summary_path, low_memory=False)
fies["PROVINCE_NAME"] = pd.to_numeric(fies["RPROV"],errors="coerce").map(FIES_PROVINCE_MAP)
fies_cal = fies[fies["PROVINCE_NAME"].notna()].copy()

print("FIES national rows:", len(fies))
print("CALABARZON rows:", len(fies_cal))
print(fies_cal["PROVINCE_NAME"].value_counts())

target_audit = []
for target in ["BREAD","VEG","FRUIT","FRUIT_VEG"]:
    y = pd.to_numeric(fies_cal[target],errors="coerce")
    target_audit.append({
        "target":target,
        "n":int(y.notna().sum()),
        "mean":y.mean(),
        "median":y.median(),
        "p95":y.quantile(.95),
        "zero_rate_pct":float(np.isclose(y.fillna(np.nan),0).mean()*100),
    })
target_audit_df = pd.DataFrame(target_audit)
display(target_audit_df)
target_audit_df.to_csv(ARTIFACT_ROOT / "reports" / "fies_target_audit.csv", index=False)


FIES national rows: 163268
CALABARZON rows: 8156
PROVINCE_NAME
Quezon      2413
Batangas    1482
Laguna      1479
Rizal       1447
Cavite      1335
Name: count, dtype: int64


,target,n,mean,median,p95,zero_rate_pct
0,BREAD,8156,26539.274440,24207.00,50349.500,0.036783
1,VEG,8156,7502.590002,6203.00,17179.375,0.478176
2,FRUIT,8156,4910.714766,3649.55,12922.500,0.220696
3,FRUIT_VEG,8156,780.054620,380.00,2886.750,23.001471


## 9. FIES demand-proxy estimation

This is **not** a time-series forecast.

The purpose of the household model is to test whether socioeconomic household attributes can estimate the FIES expenditure-category target and to produce survey-weighted province-level proxy estimates.

### Leakage controls

Excluded from predictors:
- `SEQ_NO` and identifiers
- `BREAD`, `VEG`, `FRUIT`
- `FOOD_HOME`, `FOOD`, `TOTEX`, and other totals that directly contain/reflect the food target
- target-family components

`RFACT` is used as a **survey aggregation weight**, not as the household predictor.


In [11]:

available_num = [c for c in DEMAND_NUMERIC_FEATURES if c in fies_cal.columns]
available_cat = [c for c in DEMAND_CATEGORICAL_FEATURES if c in fies_cal.columns]

def build_demand_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute",SimpleImputer(strategy="median"))]), available_num),
        ("cat", Pipeline([
            ("impute",SimpleImputer(strategy="most_frequent")),
            ("onehot",OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), available_cat),
    ], remainder="drop", sparse_threshold=0)

def build_demand_pipeline_gb(**overrides):
    params = dict(n_estimators=250, learning_rate=.05, max_depth=3, min_samples_leaf=5, random_state=RANDOM_STATE)
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", GradientBoostingRegressor(**params)),
    ])

def build_demand_pipeline_xgb(**overrides):
    params = dict(DEMAND_XGB_PARAMETERS)
    params.update(overrides)
    return Pipeline([
        ("prep", build_demand_preprocessor()),
        ("model", XGBRegressor(**params)),
    ])

def fit_weighted(pipeline, X, y, w):
    pipeline.fit(X, y, model__sample_weight=np.asarray(w, dtype=float))
    return pipeline

def weighted_mean(y,w):
    y=np.asarray(y,float); w=np.asarray(w,float)
    m=np.isfinite(y)&np.isfinite(w)&(w>0)
    return np.average(y[m],weights=w[m]) if m.any() else np.nan

def province_mean_naive(train_y, train_prov, test_prov):
    means = train_y.groupby(train_prov).mean()
    overall = float(np.nanmean(train_y.to_numpy()))
    return test_prov.map(means).fillna(overall).to_numpy()

# Broad-category proxy specificity, mirrored in commodity_demand_registry below.
DEMAND_SPECIFICITY = {
    "rice": "category_proxy",
    "vegetable_shared": "shared_broad_category",
    "banana": "category_proxy",
}

demand_model_rows=[]
demand_province_rows=[]
demand_artifacts={}

# Train only one VEG model and share it for tomato/red_onion to avoid pretending FIES distinguishes them.
target_models = {"BREAD":"rice", "VEG":"vegetable_shared", "FRUIT":"banana"}

for target, artifact_name in target_models.items():
    cols = available_num + available_cat
    assert target not in cols and "RFACT" not in cols, f"{artifact_name}: target or survey weight leaked into features"

    X = fies_cal[cols].copy()
    X["RPROV"] = X["RPROV"].astype(str)
    y = pd.to_numeric(fies_cal[target],errors="coerce")
    valid = y.notna()
    X,y = X.loc[valid],y.loc[valid]
    weights = pd.to_numeric(fies_cal.loc[valid,"RFACT"],errors="coerce").fillna(1.0)
    province = fies_cal.loc[valid,"PROVINCE_NAME"]

    tr_idx, te_idx = train_test_split(
        np.arange(len(X)), test_size=.25, random_state=RANDOM_STATE
    )
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
    wtr, wte = weights.iloc[tr_idx], weights.iloc[te_idx]
    prov_tr, prov_te = province.iloc[tr_idx], province.iloc[te_idx]
    naive_te = province_mean_naive(ytr, prov_tr, prov_te)

    def score_candidate(name, factory):
        pipeline = factory()
        fit_weighted(pipeline, Xtr, ytr, wtr)
        pred = pipeline.predict(Xte)
        unweighted = regression_metrics(yte, pred, seasonal_naive=naive_te)
        weighted = weighted_regression_metrics(yte, pred, wte)
        return {"name":name, "factory":factory, "pred":pred, "unweighted":unweighted, "weighted":weighted}

    def candidate_key(c):
        r2 = c["weighted"].get("r2", np.nan)
        wape = c["unweighted"].get("WAPE", np.nan)
        return (-(r2 if np.isfinite(r2) else -np.inf), wape if np.isfinite(wape) else np.inf)

    candidates = {
        "GradientBoostingRegressor": score_candidate("GradientBoostingRegressor", build_demand_pipeline_gb),
        "XGBRegressor": score_candidate("XGBRegressor", build_demand_pipeline_xgb),
    }

    best_name = min(candidates, key=lambda n: candidate_key(candidates[n]))
    best = candidates[best_name]
    tuning_note = "no additional tuning attempted (candidate already at/above the usable-proxy band)"

    # Light regularized-variant sweep for targets that are still weak, mirroring
    # ross'_work's philosophy of leaning on regularization for noisy FIES signal
    # rather than assuming a bug. Capped at two extra variants, kept only if better.
    if not np.isfinite(best["weighted"].get("r2", np.nan)) or best["weighted"]["r2"] < 0.45:
        tuned_variants = {
            "GradientBoostingRegressor_regularized": lambda: build_demand_pipeline_gb(
                learning_rate=0.03, min_samples_leaf=15, max_depth=2, subsample=0.8
            ),
            "XGBRegressor_regularized": lambda: build_demand_pipeline_xgb(
                max_depth=2, min_child_weight=20, reg_lambda=10.0, learning_rate=0.01
            ),
        }
        improved = False
        for vname, vfactory in tuned_variants.items():
            vcand = score_candidate(vname, vfactory)
            candidates[vname] = vcand
            if candidate_key(vcand) < candidate_key(best):
                best_name, best = vname, vcand
                improved = True
        tuning_note = (
            "attempted regularized GB/XGB variants; kept the best-scoring variant"
            if improved else
            "attempted regularized GB/XGB variants; none improved on the original candidates, original kept"
        )

    m_weighted = best["weighted"]
    m_unweighted = best["unweighted"]

    actual_w = weighted_mean(yte, wte)
    pred_w = weighted_mean(best["pred"], wte)
    aggregate_error_pct = (pred_w-actual_w)/actual_w*100 if actual_w else np.nan

    verdict, verdict_reason = demand_verdict(m_weighted.get("r2", np.nan), m_unweighted.get("WAPE", np.nan))

    # Province leave-one-out audit: tests geographic generalization, weighted fit,
    # using the exact hyperparameters of the model family that won the race above.
    province_scores=[]
    for prov in CALABARZON:
        test_mask = province.eq(prov).to_numpy()
        if test_mask.sum()==0 or (~test_mask).sum()<100:
            continue
        gm = best["factory"]()
        fit_weighted(gm, X.loc[~test_mask], y.loc[~test_mask], weights.loc[~test_mask])
        gp = gm.predict(X.loc[test_mask])
        province_scores.append({
            "province":prov,
            "R2":r2_score(y.loc[test_mask],gp) if y.loc[test_mask].std()>0 else np.nan,
            "MAE":mean_absolute_error(y.loc[test_mask],gp),
        })

    # Refit the winning family on all CALABARZON households, weighted.
    final_model = best["factory"]()
    fit_weighted(final_model, X, y, weights)
    all_pred=final_model.predict(X)
    assert np.isfinite(all_pred).all(), f"{artifact_name}: non-finite predictions after full refit"

    specificity = DEMAND_SPECIFICITY[artifact_name]
    limitations=[
        "FIES expenditure-category proxy, not physical commodity demand.",
        "Cross-sectional 2023 estimator; not an observed quarterly demand forecast.",
    ]
    if specificity == "shared_broad_category":
        limitations.append(
            "Single VEG model shared across Tomato and Red Onion; FIES does not distinguish these commodities."
        )

    art={
        "schema_version":"3.2",
        "kind":"demand_proxy_estimator",
        "target":target,
        "artifact_name":artifact_name,
        "model_id":f"{artifact_name}-demand-v1",
        "model":final_model,
        "selected_model":best_name,
        "candidate_race":{n:{"weighted":c["weighted"],"unweighted":c["unweighted"]} for n,c in candidates.items()},
        "tuning_note":tuning_note,
        "features":cols,
        "weighted_metrics":m_weighted,
        "unweighted_metrics":m_unweighted,
        "weighted_aggregate_error_pct":aggregate_error_pct,
        "province_holdout":province_scores,
        "verdict":verdict,
        "verdict_reason":verdict_reason,
        "commodity_specificity":specificity,
        "dataset_counts":{
            "train_households":int(len(tr_idx)),
            "heldout_households":int(len(te_idx)),
            "provinces":sorted(province.unique().tolist()),
        },
        "limitations":limitations,
    }
    joblib.dump(art,ARTIFACT_ROOT/"demand"/f"{artifact_name}.joblib")
    demand_artifacts[artifact_name]=art

    demand_model_rows.append({
        "target":target,"artifact":artifact_name,"selected_model":best_name,
        "R2":m_weighted.get("r2",np.nan),"MAE":m_weighted.get("mae",np.nan),"RMSE":m_weighted.get("rmse",np.nan),
        "unweighted_R2":m_unweighted.get("R2",np.nan),
        "WAPE":m_unweighted.get("WAPE",np.nan),"sMAPE":m_unweighted.get("sMAPE",np.nan),
        "MAPE":m_unweighted.get("MAPE",np.nan),"MASE":m_unweighted.get("MASE",np.nan),
        "Bias":m_unweighted.get("Bias",np.nan),"N":m_unweighted.get("N",np.nan),
        "ZeroRatePct":m_unweighted.get("ZeroRatePct",np.nan),"NonZeroN":m_unweighted.get("NonZeroN",np.nan),
        "weighted_aggregate_error_pct":aggregate_error_pct,
        "province_holdout_mean_MAE":np.mean([r["MAE"] for r in province_scores]) if province_scores else np.nan,
        "province_holdout_mean_R2":np.nanmean([r["R2"] for r in province_scores]) if province_scores else np.nan,
        "verdict":verdict,"verdict_reason":verdict_reason,
    })

    tmp=fies_cal.loc[valid,["PROVINCE_NAME","RFACT"]].copy()
    tmp["actual"]=y.to_numpy()
    tmp["predicted"]=all_pred
    for prov,g in tmp.groupby("PROVINCE_NAME"):
        demand_province_rows.append({
            "target":target,
            "province":prov,
            "actual_weighted_mean":weighted_mean(g["actual"],g["RFACT"]),
            "predicted_weighted_mean":weighted_mean(g["predicted"],g["RFACT"]),
            "households":len(g),
            "sum_weight":g["RFACT"].sum(),
        })

    print(target, "=>", best_name, verdict, "| weighted R2:", round(m_weighted.get("r2",float("nan")),3),
          "| WAPE:", round(m_unweighted.get("WAPE",float("nan")),1))

demand_model_report=pd.DataFrame(demand_model_rows)
demand_province_baseline=pd.DataFrame(demand_province_rows)

display(demand_model_report)
display(demand_province_baseline)

demand_model_report.to_csv(ARTIFACT_ROOT/"reports"/"demand_proxy_model_metrics.csv",index=False)
demand_province_baseline.to_csv(ARTIFACT_ROOT/"prepared"/"demand_proxy_province_baseline.csv",index=False)


BREAD => GradientBoostingRegressor USABLE_PROXY | weighted R2: 0.537 | WAPE: 23.3


VEG => XGBRegressor INDICATIVE_PROXY | weighted R2: 0.256 | WAPE: 42.0


FRUIT => GradientBoostingRegressor_regularized INDICATIVE_PROXY | weighted R2: 0.319 | WAPE: 48.7


,target,artifact,selected_model,R2,MAE,RMSE,unweighted_R2,WAPE,sMAPE,MAPE,MASE,Bias,N,ZeroRatePct,NonZeroN,weighted_aggregate_error_pct,province_holdout_mean_MAE,province_holdout_mean_R2,verdict,verdict_reason
0,BREAD,rice,GradientBoostingRegressor,0.537039,6112.045869,8518.199396,0.552742,23.347783,23.953507,27.860499,0.651069,146.721338,2039,0.098087,2037,1.361697,6590.364968,0.491855,USABLE_PROXY,weighted R2 and/or WAPE meet the usable-proxy bar
1,VEG,vegetable_shared,XGBRegressor,0.255827,3264.656311,4982.271649,0.271429,42.020643,42.656574,66.177447,0.841935,100.361449,2039,0.441393,2030,1.072409,3256.119866,0.211469,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...
2,FRUIT,banana,GradientBoostingRegressor_regularized,0.319276,2479.030199,3697.391525,0.334961,48.677817,50.667972,101.995880,0.789931,-47.624180,2039,0.294262,2033,0.189965,2483.616902,0.289358,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...


,target,province,actual_weighted_mean,predicted_weighted_mean,households,sum_weight
0,BREAD,Batangas,23475.038059,23739.065694,1482,8.134120e+05
1,BREAD,Cavite,25204.745147,25418.229264,1335,1.151652e+06
2,BREAD,Laguna,27124.441782,26900.421976,1479,9.975820e+05
3,BREAD,Quezon,27060.633371,26603.173573,2413,5.632850e+05
4,BREAD,Rizal,27941.670080,27968.051117,1447,7.784270e+05
5,VEG,Batangas,7053.250144,7231.760101,1482,8.134120e+05
6,VEG,Cavite,7930.968847,7966.961803,1335,1.151652e+06
7,VEG,Laguna,7901.725405,7816.896613,1479,9.975820e+05
8,VEG,Quezon,5818.826946,5994.577647,2413,5.632850e+05
9,VEG,Rizal,9128.567178,8871.811599,1447,7.784270e+05


## 9b. Demand-proxy validation reports

Per-target GradientBoosting-vs-XGBoost comparison (weighted + unweighted metrics), MASE against a province-mean naive baseline, the `weighted_aggregate_error_pct` formula spelled out, province leave-one-out results, and an honest tiered verdict (`USABLE_PROXY` / `INDICATIVE_PROXY` / `INSUFFICIENT_DATA`) -- written to `reports/{artifact}_validation_report.md` per target and consolidated into `reports/demand_validation_summary.csv`.


In [12]:

DEMAND_REPORT_LABELS = {
    "rice": "Rice (via BREAD proxy)",
    "vegetable_shared": "Vegetable -- Tomato & Red Onion (via VEG proxy)",
    "banana": "Banana (via FRUIT proxy)",
}

def _fmt(v, spec="{:.3f}"):
    return spec.format(v) if v is not None and np.isfinite(v) else "n/a"

demand_validation_rows=[]

for artifact_name, art in demand_artifacts.items():
    label = DEMAND_REPORT_LABELS.get(artifact_name, artifact_name)
    wm = art["weighted_metrics"]
    um = art["unweighted_metrics"]
    counts = art["dataset_counts"]

    # Sanity checks -- fail loud rather than silently shipping a bad artifact.
    assert set(art["features"]).isdisjoint({art["target"], "RFACT"}), \
        f"{artifact_name}: target or survey weight leaked into features"
    assert wm.get("nonfinite_predictions", 0) == 0, f"{artifact_name}: non-finite predictions in holdout"
    assert um.get("N", 0) > 0, f"{artifact_name}: empty holdout set"

    lines = []
    lines.append(f"# Demand-proxy validation report -- {label}")
    lines.append("")
    lines.append(
        f"Artifact: `demand/{artifact_name}.joblib` | Target column: `{art['target']}` | "
        f"Model selected: **{art['selected_model']}**"
    )
    lines.append("")
    lines.append("## Dataset")
    lines.append(f"- Training households: {counts['train_households']}")
    lines.append(f"- Held-out households: {counts['heldout_households']}")
    lines.append(f"- Provinces covered: {', '.join(counts['provinces'])}")
    lines.append("")
    lines.append("## Candidate comparison (weighted metrics, sample_weight = RFACT)")
    lines.append("")
    lines.append("| Candidate | weighted R2 | weighted MAE | weighted RMSE | normalized MAE % | aggregate bias % |")
    lines.append("|---|---|---|---|---|---|")
    for cname, cres in art["candidate_race"].items():
        cw = cres["weighted"]
        marker = " **(selected)**" if cname == art["selected_model"] else ""
        lines.append(
            f"| {cname}{marker} | {_fmt(cw.get('r2'))} | {_fmt(cw.get('mae'),'{:.1f}')} | "
            f"{_fmt(cw.get('rmse'),'{:.1f}')} | {_fmt(cw.get('normalized_mae_percent'),'{:.1f}')}% | "
            f"{_fmt(cw.get('aggregate_bias_percent'),'{:.2f}')}% |"
        )
    lines.append("")
    lines.append(f"Tuning note: {art['tuning_note']}")
    lines.append("")
    lines.append("## Selected model -- full metric set")
    lines.append("")
    lines.append("| Metric | Weighted (survey-representative) | Unweighted (raw holdout) |")
    lines.append("|---|---|---|")
    lines.append(f"| R2 | {_fmt(wm.get('r2'))} | {_fmt(um.get('R2'))} |")
    lines.append(f"| MAE | {_fmt(wm.get('mae'),'{:.1f}')} | {_fmt(um.get('MAE'),'{:.1f}')} |")
    lines.append(f"| RMSE | {_fmt(wm.get('rmse'),'{:.1f}')} | {_fmt(um.get('RMSE'),'{:.1f}')} |")
    lines.append(f"| WAPE | -- | {_fmt(um.get('WAPE'),'{:.2f}')}% |")
    lines.append(f"| sMAPE | -- | {_fmt(um.get('sMAPE'),'{:.2f}')}% |")
    lines.append(f"| MAPE | -- | {_fmt(um.get('MAPE'),'{:.2f}')}% |")
    lines.append(f"| MASE (vs. province-mean naive) | -- | {_fmt(um.get('MASE'),'{:.3f}')} |")
    lines.append(f"| Bias | -- | {_fmt(um.get('Bias'),'{:.1f}')} |")
    lines.append("")
    lines.append(
        "MASE is computed against a **province-mean naive baseline** (predict each "
        "held-out household's target as the training-set mean for its province), not a "
        "seasonal-lag naive, because FIES 2023 is cross-sectional with no time axis. "
        "MASE < 1 means the model beats that province-mean baseline; MASE >= 1 means it does not."
    )
    lines.append("")
    lines.append("## Population-level bias check")
    lines.append("")
    lines.append(
        f"`weighted_aggregate_error_pct` = {_fmt(art['weighted_aggregate_error_pct'],'{:.2f}')}%. "
        "Formula: `(weighted_mean(prediction, RFACT) - weighted_mean(actual, RFACT)) / "
        "weighted_mean(actual, RFACT) * 100`, computed on the holdout set only. This is a "
        "population-level bias check -- overestimates and underestimates at the household "
        "level can cancel out here -- and is not a substitute for the per-household WAPE/MASE above."
    )
    lines.append("")
    lines.append("## Province leave-one-out holdout")
    lines.append("")
    lines.append("| Province | R2 | MAE |")
    lines.append("|---|---|---|")
    for row in art["province_holdout"]:
        lines.append(f"| {row['province']} | {_fmt(row['R2'])} | {_fmt(row['MAE'],'{:.1f}')} |")
    lines.append("")
    lines.append("## Proxy-mapping caveat")
    lines.append("")
    if art["commodity_specificity"] == "shared_broad_category":
        lines.append(
            f"`{art['target']}` is a broad FIES vegetable-expenditure category shared by Tomato "
            "and Red Onion. This model does not and cannot distinguish between the two "
            "commodities; it is deployed identically under both."
        )
    else:
        commodity_word = label.split(" (")[0]
        lines.append(
            f"`{art['target']}` is a household expenditure-category proxy for **{commodity_word}**, "
            "not a measurement of physical purchase quantity of that commodity. Treat outputs as "
            "directional/indicative demand pressure, not literal commodity volume."
        )
    lines.append("")
    lines.append("## Verdict")
    lines.append("")
    lines.append(f"**{art['verdict']}** -- {art['verdict_reason']}")
    lines.append("")

    report_path = ARTIFACT_ROOT / "reports" / f"{artifact_name}_validation_report.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print("wrote", report_path)

    demand_validation_rows.append({
        "target": art["target"],
        "artifact_name": artifact_name,
        "selected_model": art["selected_model"],
        "weighted_R2": wm.get("r2"),
        "unweighted_R2": um.get("R2"),
        "WAPE": um.get("WAPE"),
        "sMAPE": um.get("sMAPE"),
        "MASE": um.get("MASE"),
        "weighted_aggregate_error_pct": art["weighted_aggregate_error_pct"],
        "verdict": art["verdict"],
        "verdict_reason": art["verdict_reason"],
    })

demand_validation_summary = pd.DataFrame(demand_validation_rows)
display(demand_validation_summary)
demand_validation_summary.to_csv(ARTIFACT_ROOT / "reports" / "demand_validation_summary.csv", index=False)


wrote artifacts_v3_2\reports\rice_validation_report.md
wrote artifacts_v3_2\reports\vegetable_shared_validation_report.md
wrote artifacts_v3_2\reports\banana_validation_report.md


,target,artifact_name,selected_model,weighted_R2,unweighted_R2,WAPE,sMAPE,MASE,weighted_aggregate_error_pct,verdict,verdict_reason
0,BREAD,rice,GradientBoostingRegressor,0.537039,0.552742,23.347783,23.953507,0.651069,1.361697,USABLE_PROXY,weighted R2 and/or WAPE meet the usable-proxy bar
1,VEG,vegetable_shared,XGBRegressor,0.255827,0.271429,42.020643,42.656574,0.841935,1.072409,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...
2,FRUIT,banana,GradientBoostingRegressor_regularized,0.319276,0.334961,48.677817,50.667972,0.789931,0.189965,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...


## 10. Commodity mapping for the demand proxy

- Rice → `BREAD`
- Banana → `FRUIT`
- Tomato → shared `VEG`
- Red Onion → shared `VEG`

The notebook deliberately **does not create two independent FIES household models for Tomato and Red Onion**, because the FIES summary provides a broad `VEG` category rather than separate tomato/onion expenditure targets.


In [13]:

commodity_demand_registry = {
    "rice":{
        "artifact":"demand/rice.joblib",
        "target":"BREAD",
        "label":"Cereal Household Demand Proxy",
        "commodity_specificity":"category_proxy",
    },
    "tomato":{
        "artifact":"demand/vegetable_shared.joblib",
        "target":"VEG",
        "label":"Vegetable Household Demand Proxy",
        "commodity_specificity":"shared_broad_category",
    },
    "red_onion":{
        "artifact":"demand/vegetable_shared.joblib",
        "target":"VEG",
        "label":"Vegetable Household Demand Proxy",
        "commodity_specificity":"shared_broad_category",
    },
    "banana":{
        "artifact":"demand/banana.joblib",
        "target":"FRUIT",
        "label":"Fruit Household Demand Proxy",
        "commodity_specificity":"category_proxy",
    },
}
(ARTIFACT_ROOT/"config"/"commodity_demand_registry.json").write_text(
    json.dumps(commodity_demand_registry,indent=2),encoding="utf-8"
)


732

## 11. Build a monthly/quarterly CALABARZON LFS economic-activity indicator

In [14]:

# LFS is used only as a temporal socioeconomic indicator.
# It is NOT treated as direct food demand.

lfs_files = [
    p for p in lfs_root.rglob("*.csv")
    if "LFS PUF" in p.name or p.name.lower().startswith("lfs20")
]

def read_lfs_indicator(path):
    try:
        header = pd.read_csv(path,nrows=0).columns.tolist()
    except Exception:
        return None

    required = {"PUFSVYYR","PUFSVYMO","PUFNEWEMPSTAT"}
    if not required.issubset(header):
        return None

    use = [c for c in ["PUFREG","PUFSVYYR","PUFSVYMO","PUFPWGTPRV","PUFNEWEMPSTAT","PUFHHSIZE"] if c in header]
    df=pd.read_csv(path,usecols=use,low_memory=False)

    # Region code 4 is used only where PUFREG is available.
    if "PUFREG" not in df:
        return None
    reg=pd.to_numeric(df["PUFREG"],errors="coerce")
    df=df[reg.eq(4)].copy()
    if df.empty:
        return None

    year=int(pd.to_numeric(df["PUFSVYYR"],errors="coerce").dropna().mode().iloc[0])
    month=int(pd.to_numeric(df["PUFSVYMO"],errors="coerce").dropna().mode().iloc[0])

    emp=pd.to_numeric(df["PUFNEWEMPSTAT"],errors="coerce")
    w=pd.to_numeric(df.get("PUFPWGTPRV",1.0),errors="coerce").fillna(1.0)

    # Coding varies by survey vintage; rather than assume employed-code semantics,
    # derive a stable "work participation share" only when exactly two/three dominant codes exist.
    code_counts=emp.value_counts(normalize=True)
    dominant=code_counts.index[:3].tolist()
    # Most LFS releases code employed persons as 1; verify coverage before using.
    employed=(emp==1).astype(float)
    employment_share=weighted_mean(employed,w)

    hhsize=pd.to_numeric(df.get("PUFHHSIZE",np.nan),errors="coerce")
    mean_hhsize=weighted_mean(hhsize,w) if hasattr(hhsize,"__len__") else np.nan

    return {
        "date":pd.Timestamp(year,month,1),
        "employment_share_proxy":employment_share,
        "mean_household_size":mean_hhsize,
        "rows":len(df),
        "source_file":path.name,
    }

lfs_rows=[]
for p in lfs_files:
    r=read_lfs_indicator(p)
    if r:
        lfs_rows.append(r)

lfs_indicator=pd.DataFrame(lfs_rows).drop_duplicates("date").sort_values("date")
if not lfs_indicator.empty:
    # Normalize around 100; used as an index, not a physical demand quantity.
    base=lfs_indicator["employment_share_proxy"].median()
    lfs_indicator["economic_activity_index"]=lfs_indicator["employment_share_proxy"]/base*100
    lfs_indicator["quarter"]=lfs_indicator["date"].dt.to_period("Q").dt.start_time
    lfs_quarterly=(
        lfs_indicator.groupby("quarter",as_index=False)["economic_activity_index"].mean()
        .rename(columns={"quarter":"date"})
    )
else:
    lfs_quarterly=pd.DataFrame(columns=["date","economic_activity_index"])

display(lfs_indicator.tail(20))
lfs_indicator.to_csv(ARTIFACT_ROOT/"prepared"/"lfs_calabarzon_activity_monthly.csv",index=False)
lfs_quarterly.to_csv(ARTIFACT_ROOT/"prepared"/"lfs_calabarzon_activity_quarterly.csv",index=False)


,date,employment_share_proxy,mean_household_size,rows,source_file,economic_activity_index,quarter
9,2022-12-01,0.423907,5.364140,2335,LFS PUF December 2022.CSV,101.029794,2022-10-01
14,2023-01-01,0.422166,5.330715,9169,LFS PUF January 2023.CSV,100.614861,2023-01-01
12,2023-02-01,0.420761,5.627379,2446,LFS PUF February 2023.CSV,100.279903,2023-01-01
26,2023-03-01,0.412850,5.351059,2330,LFS PUF March 2023.CSV,98.394451,2023-01-01
2,2023-04-01,0.424614,5.394344,9189,LFS PUF April 2023.CSV,101.198182,2023-04-01
29,2023-05-01,0.412214,5.282025,2324,LFS PUF May 2023.CSV,98.243013,2023-04-01
23,2023-06-01,0.430464,5.343078,2317,LFS PUF June 2023.CSV,102.592575,2023-04-01
19,2023-07-01,0.403143,5.223363,36008,LFS PUF July 2023.CSV,96.081130,2023-07-01
7,2023-08-01,0.426975,5.053334,2223,LFS PUF August 2023.CSV,101.761044,2023-07-01
40,2023-09-01,0.416153,4.997556,2183,LFS PUF September 2023.CSV,99.181834,2023-07-01


## 12. Derived quarterly **Demand Pressure Index**

This is the recommended temporal demand representation when no observed quarterly commodity-consumption series exists.

The index combines:

- FIES province household-expenditure baseline
- LFS regional economic-activity index
- commodity-specific seasonal price behavior
- commodity-specific seasonal supply behavior

The components are normalized first, so incompatible units are never directly added/subtracted.

**This is a proxy/index and must be labelled accordingly in the UI.**


In [15]:

def seasonal_profile_quarterly(commodity):
    # Supply seasonal profile
    s=supply_panels[commodity].dropna(subset=["target"]).copy()
    s_prof=s.groupby("season")["target"].median()
    s_prof=s_prof/s_prof.median()

    # Price -> quarterly profile
    p=price_panels[commodity].dropna(subset=["target"]).copy()
    p["q"]=p["date"].dt.quarter
    p_prof=p.groupby("q")["target"].median()
    p_prof=p_prof/p_prof.median()

    return s_prof,p_prof

# Province baseline normalized to 100 within each FIES target.
baseline=demand_province_baseline.copy()
baseline["base_index"]=baseline.groupby("target")["predicted_weighted_mean"].transform(
    lambda s:s/s.median()*100
)

pressure_rows=[]
# Build historical quarterly pressure only over the LFS indicator's available periods.
for commodity in COMMODITIES:
    target=DEMAND_TARGETS[commodity]
    b=baseline[baseline["target"].eq(target)]
    sprof,pprof=seasonal_profile_quarterly(commodity)

    for _,br in b.iterrows():
        for _,lr in lfs_quarterly.iterrows():
            q=int(pd.Timestamp(lr["date"]).quarter)
            supply_season=float(sprof.get(q,1.0))
            price_season=float(pprof.get(q,1.0))

            # Affordability is inverse price-season index.
            affordability=1.0/max(price_season,1e-6)

            # Equal-weight geometric combination avoids adding incompatible raw units.
            temporal_factor=(lr["economic_activity_index"]/100 * affordability * supply_season) ** (1/3)

            pressure_rows.append({
                "commodity":commodity,
                "province":br["province"],
                "date":lr["date"],
                "fies_base_index":br["base_index"],
                "economic_activity_index":lr["economic_activity_index"],
                "supply_season_index":supply_season*100,
                "affordability_season_index":affordability*100,
                "demand_pressure_index":br["base_index"]*temporal_factor,
            })

demand_pressure=pd.DataFrame(pressure_rows)
if not demand_pressure.empty:
    demand_pressure["demand_pressure_index"]=demand_pressure.groupby("commodity")[
        "demand_pressure_index"
    ].transform(lambda s:s/s.median()*100)

display(demand_pressure.tail(20))
demand_pressure.to_csv(ARTIFACT_ROOT/"prepared"/"quarterly_demand_pressure_index.csv",index=False)


,commodity,province,date,fies_base_index,economic_activity_index,supply_season_index,affordability_season_index,demand_pressure_index
360,banana,Quezon,2025-10-01,68.337349,99.458312,30.090587,102.734822,58.615618
361,banana,Rizal,2021-01-01,107.191954,93.260101,44.222920,99.652778,101.281899
362,banana,Rizal,2021-04-01,107.191954,98.480630,155.777080,98.883682,156.523880
363,banana,Rizal,2021-07-01,107.191954,93.349671,157.355043,100.349650,155.032961
364,banana,Rizal,2021-10-01,107.191954,98.386324,30.090587,102.734822,91.611212
365,banana,Rizal,2022-01-01,107.191954,99.075478,44.222920,99.652778,103.344794
366,banana,Rizal,2022-04-01,107.191954,99.110846,155.777080,98.883682,156.857056
367,banana,Rizal,2022-07-01,107.191954,102.209909,157.355043,100.349650,159.790435
368,banana,Rizal,2022-10-01,107.191954,104.064576,30.090587,102.734822,93.340765
369,banana,Rizal,2023-01-01,107.191954,99.763071,44.222920,99.652778,103.583317


### Demand-pressure forecast rule

Because the derived index is not observed demand, V3.2 does **not** report fake MAE/R² against it as though it were ground truth.

For an application forecast:
- use the latest FIES province baseline,
- project the temporal components,
- expose the output as **Estimated Demand Pressure Index**,
- show methodology confidence rather than “forecast accuracy”.

A simple seasonal projection can be used for the next three quarters. If later you acquire true quarterly consumption/market purchase data, replace this proxy with a trained and validated demand-forecasting target.


In [16]:

def next_quarter_dates(last_date,n=3):
    p=pd.Timestamp(last_date).to_period("Q")
    return [(p+i).start_time for i in range(1,n+1)]

future_pressure_rows=[]
if not demand_pressure.empty:
    last_date=demand_pressure["date"].max()
    future_dates=next_quarter_dates(last_date,3)

    # Economic activity: conservative recent median until a dedicated LFS forecast is justified.
    econ_recent=lfs_quarterly.tail(4)["economic_activity_index"].median() if not lfs_quarterly.empty else 100.0

    for commodity in COMMODITIES:
        target=DEMAND_TARGETS[commodity]
        b=baseline[baseline["target"].eq(target)]
        sprof,pprof=seasonal_profile_quarterly(commodity)

        for dt in future_dates:
            q=dt.quarter
            supply_season=float(sprof.get(q,1.0))
            price_season=float(pprof.get(q,1.0))
            affordability=1.0/max(price_season,1e-6)
            temporal_factor=(econ_recent/100*affordability*supply_season)**(1/3)

            for _,br in b.iterrows():
                future_pressure_rows.append({
                    "commodity":commodity,
                    "province":br["province"],
                    "date":dt,
                    "estimated_demand_pressure_index":br["base_index"]*temporal_factor,
                    "confidence":"moderate_proxy" if commodity in ["rice","banana"] else "low_to_moderate_shared_veg_proxy",
                    "observed_demand":False,
                })

future_demand_pressure=pd.DataFrame(future_pressure_rows)
if not future_demand_pressure.empty:
    future_demand_pressure["estimated_demand_pressure_index"]=future_demand_pressure.groupby("commodity")[
        "estimated_demand_pressure_index"
    ].transform(lambda s:s/s.median()*100)

display(future_demand_pressure)
future_demand_pressure.to_csv(ARTIFACT_ROOT/"prepared"/"future_demand_pressure_3q.csv",index=False)


,commodity,province,date,estimated_demand_pressure_index,confidence,observed_demand
0,rice,Batangas,2026-01-01,99.946872,moderate_proxy,False
1,rice,Cavite,2026-01-01,107.016533,moderate_proxy,False
2,rice,Laguna,2026-01-01,113.256902,moderate_proxy,False
3,rice,Quezon,2026-01-01,112.005418,moderate_proxy,False
4,rice,Rizal,2026-01-01,117.751864,moderate_proxy,False
5,rice,Batangas,2026-04-01,93.342589,moderate_proxy,False
6,rice,Cavite,2026-04-01,99.945101,moderate_proxy,False
7,rice,Laguna,2026-04-01,105.773120,moderate_proxy,False
8,rice,Quezon,2026-04-01,104.604331,moderate_proxy,False
9,rice,Rizal,2026-04-01,109.971063,moderate_proxy,False


## 13. Optional PSA commodity-flow context

In [17]:

# Optional. This section is deliberately not required for model training.
# It can enrich market opportunity with inflow/outflow dependence if internet access is available.

FLOW_URL = "https://openstat.psa.gov.ph/PXWeb/api/v1/en/DB/2L/DTR/QT/0032L4DRQD0.px"

def flow_status_note():
    return {
        "status":"optional_external_indicator",
        "use":"market opportunity / inflow-outflow context",
        "warning":"Section II - Vegetable products is broad unless lower-level PSCC items are explicitly queried.",
        "unit_policy":"convert source volume to metric tons only after confirming source unit metadata",
    }

(ARTIFACT_ROOT/"config"/"commodity_flow_methodology.json").write_text(
    json.dumps(flow_status_note(),indent=2),encoding="utf-8"
)
print(flow_status_note())


{'status': 'optional_external_indicator', 'use': 'market opportunity / inflow-outflow context', 'warning': 'Section II - Vegetable products is broad unless lower-level PSCC items are explicitly queried.', 'unit_policy': 'convert source volume to metric tons only after confirming source unit metadata'}


## 14. Opportunity-index inputs — normalized, never mixed raw units

In [18]:

# The notebook exports components, not a falsely learned "opportunity" target.
# Initial weights remain configurable/expert-defined until observed market outcomes exist.

opportunity_config={
    "schema_version":"3.2",
    "components":{
        "demand_pressure_index":{"weight":0.35,"direction":"higher_better"},
        "supply_gap_or_scarcity_index":{"weight":0.25,"direction":"higher_better"},
        "price_opportunity_index":{"weight":0.20,"direction":"higher_better"},
        "market_flow_dependence_index":{"weight":0.10,"direction":"higher_better","optional":True},
        "forecast_confidence_index":{"weight":0.10,"direction":"higher_better"},
    },
    "policy":[
        "Normalize all inputs to a common 0-100 scale before combination.",
        "Do not subtract FIES pesos directly from supply metric tons.",
        "Weights are transparent configuration, not learned ground truth.",
    ],
}
(ARTIFACT_ROOT/"config"/"opportunity_scoring_config.json").write_text(
    json.dumps(opportunity_config,indent=2),encoding="utf-8"
)


797

## 15. Unified deployment summary

In [19]:

deployment = pd.concat([
    supply_verdicts.assign(component="supply"),
    price_verdicts.assign(component="price"),
], ignore_index=True, sort=False)

# Add demand-proxy deployment status separately because its validation semantics differ.
demand_rows=[]
for c in COMMODITIES:
    target=DEMAND_TARGETS[c]
    report=demand_model_report[demand_model_report["target"].eq(target)].iloc[0]
    specificity=commodity_demand_registry[c]["commodity_specificity"]
    demand_rows.append({
        "commodity":c,
        "component":"demand_proxy",
        "selected_model":report["selected_model"],
        "R2":report["R2"],
        "MAE":report["MAE"],
        "WAPE":report["WAPE"],
        "MASE":report["MASE"],
        "weighted_aggregate_error_pct":report["weighted_aggregate_error_pct"],
        "verdict":report["verdict"],
        "reason":report["verdict_reason"],
        "specificity":specificity,
    })

deployment_all=pd.concat([deployment,pd.DataFrame(demand_rows)],ignore_index=True,sort=False)
display(deployment_all)
deployment_all.to_csv(ARTIFACT_ROOT/"reports"/"unified_deployment_summary.csv",index=False)


,commodity,kind,selected_model,MAE,MSE,RMSE,R2,MAPE,sMAPE,WAPE,...,NonZeroN,baseline_sMAPE,improvement_vs_naive_pct,rolling_median_sMAPE,rolling_worst_sMAPE,verdict,reason,component,weighted_aggregate_error_pct,specificity
0,rice,supply,hist_gradient_boosting,739.047069,3.680103e+06,1918.359430,0.983102,5.416628,5.426247,6.170531,...,20.0,22.626605,76.018288,7.525730,58.359334,CAUTION,rolling variability is elevated,supply,NaN,NaN
1,tomato,supply,random_forest,106.923047,7.352388e+04,271.152865,0.969183,27.248317,21.473553,14.063160,...,20.0,28.999335,25.951568,26.481272,91.425107,CAUTION,rolling variability is elevated,supply,NaN,NaN
2,red_onion,supply,seasonal_naive,1.261071,8.569416e+00,2.927357,0.515448,1244.634380,64.745796,50.477470,...,15.0,64.745796,0.000000,33.139871,88.714852,INSUFFICIENT_DATA,forecast errors remain too high,supply,NaN,NaN
3,banana,supply,seasonal_naive,572.249000,1.307150e+06,1143.306509,0.934330,14.704234,18.967574,12.870320,...,20.0,18.967574,0.000000,7.071017,24.433028,CAUTION,MASE=1.00 indicates no improvement over the se...,supply,NaN,NaN
4,rice,price,hist_gradient_boosting,1.833069,6.606344e+00,2.570281,0.676917,10.084788,10.335733,10.400240,...,59.0,35.286311,70.708945,5.738461,7.285104,PASS,,price,NaN,NaN
5,tomato,price,random_forest,14.330404,5.101845e+02,22.587264,0.060465,38.352551,29.455760,32.377939,...,45.0,47.790878,38.365309,32.184053,57.352353,CAUTION,rolling variability is elevated,price,NaN,NaN
6,red_onion,price,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_DATA,no candidate could be trained,price,NaN,NaN
7,banana,price,random_forest,3.821376,2.831730e+01,5.321401,-0.096755,15.204443,14.447917,14.905999,...,60.0,20.874724,30.787504,5.777772,17.560057,PASS,,price,NaN,NaN
8,rice,NaN,GradientBoostingRegressor,6112.045869,NaN,NaN,0.537039,NaN,NaN,23.347783,...,NaN,NaN,NaN,NaN,NaN,USABLE_PROXY,weighted R2 and/or WAPE meet the usable-proxy bar,demand_proxy,1.361697,category_proxy
9,tomato,NaN,XGBRegressor,3264.656311,NaN,NaN,0.255827,NaN,NaN,42.020643,...,NaN,NaN,NaN,NaN,NaN,INDICATIVE_PROXY,weak explanatory power; use for relative/direc...,demand_proxy,1.072409,shared_broad_category


## 16. Data-quality and methodology registry

In [20]:

methodology={
    "version":"3.2",
    "supply":{
        "frequency":"quarterly",
        "target":"PSA volume of production",
        "auxiliary":["area planted/harvested","yield proxy","lagged farmgate price"],
    },
    "price":{
        "frequency":"monthly",
        "target":"PSA farmgate price",
        "critical_cleaning":"non-positive prices treated as missing",
        "auxiliary":["lagged supply","lagged area","lagged yield proxy"],
    },
    "demand":{
        "fies_year":"2023 in supplied household-summary file",
        "targets":{"rice":"BREAD","tomato":"VEG","red_onion":"VEG","banana":"FRUIT"},
        "aggregation":"survey-weighted using RFACT",
        "training":"sample_weight=RFACT passed directly into model.fit(), matching EF_agriwise/ml/demand/pipeline's FIES/XGBoost reference pattern",
        "model_race":"GradientBoostingRegressor vs XGBRegressor (ross'_work reference hyperparameters) selected per target by weighted R2, tie-break on WAPE; a light regularized-variant sweep is attempted only when the winner is still below the usable-proxy band",
        "mase_baseline":"province-mean naive (no time axis exists in cross-sectional FIES 2023, so this replaces the seasonal-lag naive used for supply/price)",
        "verdict_bands":"USABLE_PROXY (weighted R2>=0.45 or WAPE<=30%), INDICATIVE_PROXY (weighted R2>=0.20 or WAPE<=55%), INSUFFICIENT_DATA otherwise",
        "temporal_proxy":"FIES baseline + LFS economic activity + seasonal price/supply indices",
        "not_claimed":["physical MT demand","observed quarterly commodity consumption"],
    },
    "validation":{
        "supply_price":"chronological holdout + rolling origin",
        "demand_household_proxy":"household holdout + province leave-one-out + weighted aggregate error + per-target validation_report.md",
    },
}
(ARTIFACT_ROOT/"reports"/"methodology_registry.json").write_text(
    json.dumps(methodology,indent=2),encoding="utf-8"
)
print(json.dumps(methodology,indent=2))


{
  "version": "3.2",
  "supply": {
    "frequency": "quarterly",
    "target": "PSA volume of production",
    "auxiliary": [
      "area planted/harvested",
      "yield proxy",
      "lagged farmgate price"
    ]
  },
  "price": {
    "frequency": "monthly",
    "target": "PSA farmgate price",
    "critical_cleaning": "non-positive prices treated as missing",
    "auxiliary": [
      "lagged supply",
      "lagged area",
      "lagged yield proxy"
    ]
  },
  "demand": {
    "fies_year": "2023 in supplied household-summary file",
    "targets": {
      "rice": "BREAD",
      "tomato": "VEG",
      "red_onion": "VEG",
      "banana": "FRUIT"
    },
    "aggregation": "survey-weighted using RFACT",
    "training": "sample_weight=RFACT passed directly into model.fit(), matching EF_agriwise/ml/demand/pipeline's FIES/XGBoost reference pattern",
    "model_race": "GradientBoostingRegressor vs XGBRegressor (ross'_work reference hyperparameters) selected per target by weighted R2, tie-brea

## 17. Application integration rules

### Supply/price

Use a numerical forecast only when the artifact verdict is `PASS` or `CAUTION`.

If `INSUFFICIENT_DATA`:
- keep the commodity visible,
- show latest observed value/history,
- display **Forecast unavailable — Insufficient Data**.

### Demand

Recommended UI label:

> **Estimated Demand Pressure Index**

Subtext:

> FIES/LFS-derived household market-demand proxy. Index values are relative and are not metric tons of consumption.

For Tomato and Red Onion, additionally state that the underlying household proxy is the broad FIES **Vegetable** expenditure category.

### Opportunity

Use normalized component indices. Never calculate:

`FIES expenditure pesos − supply metric tons`.

Those units are not commensurable.


## 18. Files to send back after Run All

Please send these for final review:

```text
artifacts_v3_2/reports/
├── supply_data_audit.csv
├── price_data_audit.csv
├── supply_deployment_verdicts.csv
├── price_deployment_verdicts.csv
├── demand_proxy_model_metrics.csv
├── unified_deployment_summary.csv
└── methodology_registry.json

artifacts_v3_2/prepared/
├── demand_proxy_province_baseline.csv
├── quarterly_demand_pressure_index.csv
└── future_demand_pressure_3q.csv
```

The first thing to check is whether the **Red Onion price** result improves after the zero-price correction and whether the FIES `BREAD`/`VEG`/`FRUIT` proxy models remain stable under province holdout.


## 19. Promote artifacts for the EF_agriwise forecast service

Copies trained bundles into `EF_agriwise/ml/artifacts/{demand,supply,price}/<slug>.joblib` with Title-Case `commodity`, `model_id`, `verdict`, `metrics`, and `limitations`, so `EF_agriwise/ml/forecasting/artifact_registry.py` can load them directly. Skipped gracefully (with a printed message) if `EF_agriwise` isn't found next to this notebook.


In [21]:

def promote_bundle(component, out_dir, filename_stem, bundle):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{filename_stem}.joblib"
    joblib.dump(bundle, out_path)
    print("promoted", component, bundle.get("commodity"), "->", out_path)
    return out_path

if EF_ARTIFACTS_ROOT is None:
    print("Skipping artifact promotion: EF_agriwise was not found relative to this notebook.")
else:
    for component in ["supply", "price"]:
        out_dir = EF_ARTIFACTS_ROOT / component
        for path in sorted((ARTIFACT_ROOT / component).glob("*.joblib")):
            bundle = joblib.load(path)
            if "commodity" not in bundle or "model_id" not in bundle or "verdict" not in bundle:
                print("skipping incompatible bundle (missing commodity/model_id/verdict):", path)
                continue
            promote_bundle(component, out_dir, slugify_commodity(bundle["commodity"]), bundle)

    demand_out_dir = EF_ARTIFACTS_ROOT / "demand"
    for artifact_name, commodities in DEMAND_ARTIFACT_COMMODITIES.items():
        src = ARTIFACT_ROOT / "demand" / f"{artifact_name}.joblib"
        if not src.exists():
            print("skipping missing demand artifact:", src)
            continue
        base = joblib.load(src)
        for commodity_title in commodities:
            promoted = dict(base)
            promoted["commodity"] = commodity_title
            promoted["metrics"] = base.get("weighted_metrics", {})
            promote_bundle("demand", demand_out_dir, slugify_commodity(commodity_title), promoted)

    print("Artifact promotion complete ->", EF_ARTIFACTS_ROOT)


promoted supply Banana -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\supply\banana.joblib
promoted supply Red Onion -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\supply\red_onion.joblib


promoted supply Rice -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\supply\rice.joblib


promoted supply Tomato -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\supply\tomato.joblib


promoted price Banana -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\price\banana.joblib


promoted price Rice -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\price\rice.joblib


promoted price Tomato -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\price\tomato.joblib
promoted demand Rice -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\demand\rice.joblib
promoted demand Tomato -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\demand\tomato.joblib


promoted demand Red Onion -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\demand\red_onion.joblib
promoted demand Banana -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts\demand\banana.joblib
Artifact promotion complete -> C:\Users\jramo\Downloads\DOST_Agriwise\EF_agriwise\ml\artifacts
